# Mi perfilado de referencias GTFS — primer incremento: agency

Preparé este notebook para leer Bronze y el encabezado de agency.txt en Staging. No escribe tablas, archivos, auditorías ni cuarentena, y no cambia los valores originales. Revisaré routes después de entender agency.

Si elijo esta alternativa PySpark, empezaré por los bloques 1 y 2 para comprobar snapshot, esquema y muestra. Después interpretaré las métricas de 3 y 4 y repetiré con 2026, cambiando versión y hash juntos.

- 2025-10-28: `07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952`.
- 2026-09-29: `08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097`.

Ya comprobé con SQL siete registros por snapshot; este notebook PySpark aún no se ha ejecutado. Para 2025 también revisé la unicidad de los IDs; todavía no extiendo esa conclusión a 2026.


In [ ]:
# BLOQUE 1 — Parámetros de lectura y selección explícita del snapshot

from datetime import date, datetime, timezone
import re

from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import (
    BooleanType, LongType, StringType, StructField, StructType,
)


def validate_profile_parameters(
    catalog_name: str,
    schema_name: str,
    staging_name: str,
    version: str,
    content_hash: str,
) -> None:
    """Valida parámetros antes de construir identificadores o rutas de lectura."""
    for label, value in (
        ("catalog", catalog_name), ("schema", schema_name), ("staging_volume", staging_name),
    ):
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
            raise ValueError(f"{label}: usa un nombre simple con letras, números y guion bajo.")
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", version):
        raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
    try:
        date.fromisoformat(version)
    except ValueError as error:
        raise ValueError("feed_version no representa una fecha válida.") from error
    if not re.fullmatch(r"[0-9a-f]{64}", content_hash):
        raise ValueError("snapshot_sha256 debe contener 64 caracteres hexadecimales.")


# Defaults del entorno actual; cambian por widgets, no dentro de las funciones.
dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("schema", "gtfs_bogota", "Schema")
dbutils.widgets.text("staging_volume", "staging_gtfs", "Volume Staging")
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
staging_volume = dbutils.widgets.get("staging_volume").strip()
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()
validate_profile_parameters(catalog, schema, staging_volume, feed_version, snapshot_sha256)

agency_table = f"`{catalog}`.`{schema}`.`bronze_gtfs_agency`"
agency_source_path = (
    f"/Volumes/{catalog}/{schema}/{staging_volume}/gtfs_static/"
    f"feed_version={feed_version}/sha256={snapshot_sha256}/agency.txt"
)
profiled_at_utc = datetime.now(timezone.utc).isoformat()
print(f"Tabla: {agency_table}")
print(f"Versión: {feed_version}; SHA-256: {snapshot_sha256}")
print(f"Perfilado iniciado en UTC: {profiled_at_utc}")

print(f"Versión de Spark: {spark.version}")


In [ ]:
# BLOQUE 2 — Evidencia de origen, esquema Bronze y muestra de agency

import csv


def read_agency_header(source_path: str) -> list[str]:
    """Lee el encabezado original sin convertir ni normalizar sus nombres."""
    with open(source_path, "r", encoding="utf-8-sig", newline="") as source_file:
        header = next(csv.reader(source_file), None)
    if not header or any(not name.strip() for name in header):
        raise RuntimeError("agency.txt tiene un encabezado vacío o inválido; revisa Staging.")
    if len(header) != len(set(header)):
        raise RuntimeError("agency.txt contiene nombres de columnas duplicados.")
    return header


if not spark.catalog.tableExists(agency_table):
    raise RuntimeError(f"No existe {agency_table}; verifica la carga Bronze y los parámetros.")

technical_columns = {
    "source_feed_version", "source_sha256", "source_file_path", "bronze_loaded_at_utc",
}
agency_bronze_df = spark.table(agency_table)
missing_technical = technical_columns - set(agency_bronze_df.columns)
if missing_technical:
    raise RuntimeError(f"Bronze no conserva la procedencia esperada: {sorted(missing_technical)}")

# Leemos primero el hash para no ocultar versiones nulas o inconsistentes.
agency_hash_df = agency_bronze_df.filter(F.col("source_sha256") == snapshot_sha256)
hash_metrics = agency_hash_df.agg(
    F.count(F.lit(1)).alias("rows"),
    F.count(F.when(
        F.col("source_feed_version").isNull()
        | (F.col("source_feed_version") != feed_version), 1,
    )).alias("wrong_version_rows"),
    F.count(F.when(
        F.col("source_file_path").isNull()
        | (F.col("source_file_path") != agency_source_path), 1,
    )).alias("wrong_source_path_rows"),
).first().asDict()
if hash_metrics["rows"] == 0:
    raise RuntimeError("No hay filas para ese hash. Comprueba versión y hash antes de continuar.")
if hash_metrics["wrong_version_rows"] or hash_metrics["wrong_source_path_rows"]:
    raise RuntimeError("La versión o ruta de procedencia no coincide; revisa parámetros y auditoría 07.")

agency_snapshot_df = agency_hash_df.filter(F.col("source_feed_version") == feed_version)
source_columns = read_agency_header(agency_source_path)
if technical_columns & set(source_columns):
    raise RuntimeError("El archivo fuente colisiona con columnas técnicas; requiere revisión.")
if set(source_columns) - set(agency_snapshot_df.columns):
    raise RuntimeError("Bronze no conserva todas las columnas del archivo agency.txt.")

bronze_business_columns = sorted(set(agency_snapshot_df.columns) - technical_columns)
non_string_columns = [
    field.name for field in agency_snapshot_df.schema.fields
    if field.name in bronze_business_columns and not isinstance(field.dataType, StringType)
]
if non_string_columns:
    raise RuntimeError(f"Se esperaba texto fuente en Bronze. Revisar tipos de: {non_string_columns}")

print(f"Filas del snapshot: {hash_metrics['rows']}")
print(f"Encabezado publicado: {source_columns}")
print("Esquema actual de Bronze, incluidas columnas técnicas:")
agency_snapshot_df.printSchema()
# Muestra acotada: no trasladamos todas las entidades del lakehouse al driver.
display(agency_snapshot_df.select(*source_columns).orderBy(*source_columns).limit(20))


## Mi comprobación antes de seguir

Voy a revisar que versión, hash y ruta coincidan, y después miraré las filas y los tipos. En Bronze espero campos fuente string porque decidí conservarlos así; ese almacenamiento no aprueba por sí solo el contrato Silver.

Luego revisaré los bloques 3 y 4. Una columna puede existir en la tabla histórica y haber estado ausente en el archivo de 2025. Para separar esas situaciones necesito el encabezado del snapshot.


In [ ]:
# BLOQUE 3 — Perfil por columna y propuesta semántica de tipos

# Estos campos conocidos son texto por su significado, no por adivinar sus valores.
# Un campo nuevo no incluido aquí quedará pendiente de revisión.
agency_field_contract = {
    "agency_id": ("string", "condicional", "Identificador; conservar ceros y caracteres."),
    "agency_name": ("string", "requerido", "Nombre publicado; no es una clave."),
    "agency_url": ("string", "requerido", "URL de la agencia; formato por revisar."),
    "agency_timezone": ("string", "requerido", "Nombre de zona horaria; no es un timestamp."),
    "agency_lang": ("string", "opcional", "Código de idioma."),
    "agency_phone": ("string", "opcional", "Teléfono; conservar signos y formato."),
    "agency_fare_url": ("string", "opcional", "URL tarifaria; medir contenido antes de usar."),
    "agency_email": ("string", "opcional", "Contacto publicado, cuando exista."),
}


def profile_agency_fields(df: DataFrame, published_columns: list[str]) -> DataFrame:
    """Calcula métricas exactas de texto sin modificar el DataFrame de entrada."""
    fields = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    columns = sorted((set(df.columns) - technical_columns) | set(agency_field_contract))
    expressions = [F.count(F.lit(1)).alias("total_rows")]
    for index, name in enumerate(columns):
        if name not in fields:
            continue
        value = F.col(name)
        non_blank = value.isNotNull() & ~value.rlike(r"^\s*$")
        trimmed = F.regexp_replace(value, r"^\s+|\s+$", "")
        expressions.extend([
            F.count(F.when(value.isNull(), 1)).alias(f"c{index}_null"),
            F.count(F.when(value == "", 1)).alias(f"c{index}_empty"),
            F.count(F.when(value.rlike(r"^\s+$"), 1)).alias(f"c{index}_spaces"),
            F.count(F.when(non_blank, 1)).alias(f"c{index}_filled"),
            F.countDistinct(F.when(non_blank, value)).alias(f"c{index}_distinct"),
            F.count(F.when(non_blank & (value != trimmed), 1)).alias(f"c{index}_edges"),
        ])
    # Solo traemos una fila de métricas agregadas, no la tabla fuente.
    metrics = df.agg(*expressions).first().asDict()
    rows = []
    for index, name in enumerate(columns):
        proposal, requiredness, reason = agency_field_contract.get(
            name, (None, "pendiente", "Campo no contemplado: revisar antes de asignar tipo."),
        )
        counts = tuple(
            metrics.get(f"c{index}_{suffix}")
            for suffix in ("null", "empty", "spaces", "filled", "distinct", "edges")
        )
        rows.append((
            name, name in published_columns, fields.get(name), proposal, requiredness,
            metrics["total_rows"], *counts, reason,
        ))
    result_schema = StructType([
        StructField("field_name", StringType(), False),
        StructField("source_column_present", BooleanType(), False),
        StructField("bronze_type", StringType(), True),
        StructField("proposed_silver_type", StringType(), True),
        StructField("requiredness", StringType(), False),
        StructField("snapshot_rows", LongType(), False),
        StructField("null_count", LongType(), True),
        StructField("empty_count", LongType(), True),
        StructField("whitespace_only_count", LongType(), True),
        StructField("non_blank_count", LongType(), True),
        StructField("distinct_non_blank_count", LongType(), True),
        StructField("edge_whitespace_count", LongType(), True),
        StructField("type_reason", StringType(), False),
    ])
    return df.sparkSession.createDataFrame(rows, schema=result_schema)


agency_column_profile_df = profile_agency_fields(agency_snapshot_df, source_columns)
display(agency_column_profile_df.orderBy("field_name"))
print("Nulo, vacío, solo espacios y no vacío son categorías exclusivas por columna presente.")
print("Los distintos y los espacios en extremos son diagnósticos adicionales; no se suman a esas categorías.")
print("Si source_column_present=false, los nulos históricos no son valores publicados por esa versión.")
print("Si el CSV ya convirtió vacíos a null, esta lectura Bronze no reconstruye esa distinción original.")


In [ ]:
# BLOQUE 4 — Duplicados de negocio, claves repetidas y posibles colisiones


def duplicate_agency_ids(df: DataFrame, published_columns: list[str]) -> DataFrame:
    """Distingue repeticiones idénticas de conflictos para IDs presentes y no vacíos."""
    business = df.select(*published_columns)
    valid_id = F.col("agency_id").isNotNull() & ~F.col("agency_id").rlike(r"^\s*$")
    return (
        business.filter(valid_id)
        .groupBy("agency_id")
        .agg(
            F.count(F.lit(1)).alias("row_count"),
            F.countDistinct(F.struct(*[F.col(name) for name in published_columns])).alias("business_variants"),
        )
        .filter(F.col("row_count") > 1)
        .withColumn(
            "observation",
            F.when(F.col("business_variants") == 1, "copias_identicas").otherwise("conflicto_de_clave"),
        )
    )


# La comparación excluye timestamps y procedencia técnica: solo compara negocio.
agency_business_df = agency_snapshot_df.select(*source_columns)
identical_rows_df = (
    agency_business_df.groupBy(*source_columns).count()
    .filter(F.col("count") > 1)
)
print("Grupos de filas fuente idénticas; una tabla vacía significa que no se encontraron:")
display(identical_rows_df.orderBy(F.col("count").desc()).limit(20))

if "agency_id" in source_columns:
    print("IDs repetidos: business_variants=1 indica copias; >1 indica atributos distintos:")
    display(duplicate_agency_ids(agency_snapshot_df, source_columns).orderBy("agency_id").limit(20))
    valid_id = F.col("agency_id").isNotNull() & ~F.col("agency_id").rlike(r"^\s*$")
    id_metrics_df = agency_business_df.agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.count(F.when(~valid_id, 1)).alias("blank_id_rows"),
        F.countDistinct(F.when(valid_id, F.col("agency_id"))).alias("distinct_non_blank_ids"),
    )
    display(id_metrics_df)
    # Mostrar una colisión potencial no aprueba aplicar trim al identificador.
    trim_collisions_df = (
        agency_business_df.filter(valid_id)
        .groupBy(F.regexp_replace("agency_id", r"^\s+|\s+$", "").alias("id_after_trim"))
        .agg(F.countDistinct("agency_id").alias("original_id_variants"))
        .filter(F.col("original_id_variants") > 1)
    )
    print("Posibles colisiones si se eliminaran espacios extremos del ID; no se aplica esa limpieza:")
    display(trim_collisions_df.orderBy("id_after_trim").limit(20))
else:
    print("agency_id no está publicado. Revisaremos cardinalidad y reglas GTFS antes de definir la clave.")

print("Fin del perfilado de agency. Revisa las salidas antes de definir reglas o transformar.")


## Qué voy a revisar y dejar registrado

Empezaré por el conteo, printSchema y la muestra de 2025 del bloque 2. Cuando pueda explicar esa salida, revisaré el perfil por columna del bloque 3 y los duplicados del bloque 4. Una tabla de duplicados vacía es un resultado válido. Después repetiré con la versión y el hash de 2026.

La propuesta de tipos es provisional y depende del significado: agency_id y agency_phone conservarían string aunque parezcan números. Los campos nuevos necesitan otra decisión. Este perfil todavía no valida todos los formatos de URL, zona horaria o idioma ni las relaciones con routes.

Fuentes: [GTFS agency.txt](https://gtfs.org/documentation/schedule/reference/#agencytxt), [count_distinct](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.count_distinct.html) y [groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html).
